In [17]:
import ollama
import faiss
import chardet
from langchain.text_splitter import CharacterTextSplitter
from langchain_community.vectorstores.faiss import FAISS
from langchain_ollama import OllamaEmbeddings

# Detectar codificación
file_path = "documento.txt"
with open(file_path, "rb") as raw_file:
    result = chardet.detect(raw_file.read())
    detected_encoding = result['encoding']
    print(f"Codificación detectada: {detected_encoding}")

# Leer usando la codificación detectada
with open(file_path, "r", encoding=detected_encoding, errors='replace') as file:
    text = file.read()


# Función personalizada para dividir el texto en fragmentos manejables
def custom_text_splitter(text, chunk_size, chunk_overlap):
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        start += chunk_size - chunk_overlap
    return chunks

# Dividir el texto en fragmentos manejables
chunks = custom_text_splitter(text, chunk_size=800, chunk_overlap=100)



# Configurar embeddings usando Ollama con el modelo 'nomic-embed-text'
embeddings = OllamaEmbeddings(model="all-minilm:33m")

# Crear la base de datos vectorial FAISS
vectorstore = FAISS.from_texts(chunks, embeddings)
print("Fragmentos almacenados en la base de datos vectorial FAISS.")

# Función para consultar con Mistral, asegurando que solo use el contexto proporcionado
def query_ollama_with_context(context, question):
    if not context.strip():  # Si el contexto está vacío, evitar respuestas fuera del documento
        return "Esa información no está en nuestra base de datos."

    prompt = f"""
        Responde exclusivamente con base en el siguiente contexto.
         
        Contexto:
        {context}
         
        Si la información no está presente en el contexto, responde: 
        "No tengo suficiente información para responder con certeza."
         
        Pregunta:
        {question}
         
        Respuesta:
    """

  

    response = ollama.chat(
        model='llama3.2',
        messages=[{'role': 'user', 'content': prompt}]
    )
    return response['message']['content']

# Recuperar fragmentos relevantes
def retrieve_relevant_chunks(query, vectorstore):
    results = vectorstore.similarity_search(query, k=3)
    #results = vectorstore.similarity_search_with_score(query=question, k=3)
    #filtered_context = [doc.page_content for doc, score in results if score > 0.5]


    return results if results else []
    



# Pipeline RAG con validación estricta de relevancia
def rag_pipeline(question, vectorstore):
    results = retrieve_relevant_chunks(question, vectorstore)
    
    if not results:
        return  print("Esa información no está en el documento cargado, pero basado en la información que tengo la siguiente respuesta te puede ser útil\n")
    
    context = "\n".join([result.page_content for result in results])
    
    response = query_ollama_with_context(context, question)
    return response


Codificación detectada: utf-8
Fragmentos almacenados en la base de datos vectorial FAISS.


In [19]:
# Ejemplo de consulta válida
question = "Cuantos pokemones existen?"
response = rag_pipeline(question, vectorstore)
print("Dada esta pregunta:",question,"\n","La respuesta es:","\n", response)

Dada esta pregunta: Cuantos pokemones existen? 
 La respuesta es: 
 Según el contexto proporcionado, se menciona que esta generación de Pokémon abarca desde 151 Pokémon (de Bulbasaur a Mew) y que la última generación anterior incluye un total de 721 Pokémon. Sin embargo, no se menciona explícitamente cuántos Pokémon existen en general.

La respuesta más precisa sería:

"No tengo suficiente información para responder con certeza."


In [ ]:

# Ejemplo de consulta fuera del documento
question = "¿Cuáles son los planetas más cercanos a la Tierra?"
response = rag_pipeline(question, vectorstore)
print("Dada esta pregunta:",question,"\n","La respuesta es:", response)

In [ ]:

# Ejemplo de consulta fuera del documento
question = "¿Qué animales viven en la amazonía?"
response = rag_pipeline(question, vectorstore)
print("Dada esta pregunta:",question,"\n","La respuesta es:", response)

In [ ]:
question = "¿cuales son las responsabilidades del product owner?"
response = rag_pipeline(question, vectorstore)
print("Dada esta pregunta:",question,"\n","La respuesta es:", response)

In [ ]:
question = "¿Que hace el equipo scrum?"
response = rag_pipeline(question, vectorstore)
print("Dada esta pregunta:",question,"\n","La respuesta es:", response)

In [ ]:
question = "¿que hacen los equipos de desarrollo o developers?"
response = rag_pipeline(question, vectorstore)
print("Dada esta pregunta:",question,"\n","La respuesta es:", response)

In [ ]:
question = "¿que alimentos comen las hormigas?"
response = rag_pipeline(question, vectorstore)
print("Dada esta pregunta:",question,"\n","La respuesta es:", response)

In [ ]:
question = "¿que es el daily o scrum diario?"
response = rag_pipeline(question, vectorstore)
print("Dada esta pregunta:",question,"\n","La respuesta es:", response)

In [ ]:
question = "¿que es un sprint y cuanto es la duración de un sprint?"
response = rag_pipeline(question, vectorstore)
print("Dada esta pregunta:",question,"\n","La respuesta es:", response)

In [ ]:
question = "¿que es una retrospective o retrospectiva?"
response = rag_pipeline(question, vectorstore)
print("Dada esta pregunta:",question,"\n","La respuesta es:", response)

In [ ]:
question = "¿que es scrum?"
response = rag_pipeline(question, vectorstore)
print("Dada esta pregunta:",question,"\n","La respuesta es:", response)

In [ ]:
question = "¿Scrum sirve para gestionar proyectos en cascada?"
response = rag_pipeline(question, vectorstore)
print("Dada esta pregunta:",question,"\n","La respuesta es:", response)

In [ ]:
question = "¿Scrum sirve para comprar caballos?"
response = rag_pipeline(question, vectorstore)
print("Dada esta pregunta:",question,"\n","La respuesta es:", response)